# Variational Autoencoders (VAEs) 

VAEs can be a better fit than standard autoencoders when we need to profile normal network communication and share/update the model across distributed locations.

In [ ]:
%pip install torch scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/204.1 MB ? eta -:--:--
   --- ------------------------------------ 16.0/204.1 MB 83.7 MB/s eta 0:00:03
   ------- -------------------------------- 38.5/204.1 MB 94.0 MB/s eta 0:00:02
   ----------- ---------------------------- 59.0/204.1 MB 94.0 MB/s eta 0:00:02
   --------------- ------------------------ 79.4/204.1 MB 93.8 MB/s eta 0:00:02
   ------------------ --------------------- 95.4/204.1 MB 89.5 MB/s eta 0:00:02
   --------------------- ----------------- 110.9/204.1 MB 87.4 MB/s eta 0:00:02
   ------------------------ -------------- 126.6/204.1 MB 86.0 MB/s eta 0:00:01
   --------------------------- ----------- 142.3/204.1 MB 84.2 MB/s eta 0:00:01
   ------------------------------ -------- 160.4/204.1 MB 84.0 MB/s eta 0:00:01
   ---------------------------------- ---- 178.8/204.1 MB 84.0 MB/s eta 0:00:01
   ------------------------------------- - 196.3/204.1 MB 83.7 MB/s eta 0:00:01
   --------------------------------------  203.9/

## Sample data

TODO: take our data as input data and rewrite to the same way as Autoencoder module.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==== Simulated NetFlow-like Data ====
# Columns: duration, packets, bytes, src_port, dst_port, protocol
df = pd.DataFrame({
    'duration': np.random.exponential(scale=1.0, size=1000),
    'packets': np.random.poisson(lam=10, size=1000),
    'bytes': np.random.normal(loc=5000, scale=1000, size=1000),
    'src_port': np.random.randint(1024, 65535, size=1000),
    'dst_port': np.random.choice([80, 443, 53, 22], size=1000),
    'protocol': np.random.choice([6, 17], size=1000)  # TCP=6, UDP=17
})

# Normalize
scaler = StandardScaler()
X = scaler.fit_transform(df.values)
X_tensor = torch.tensor(X, dtype=torch.float32)

dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
input_dim = X.shape[1]
latent_dim = 2


## VAE Model

In [10]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc21 = nn.Linear(64, latent_dim)  # Mean
        self.fc22 = nn.Linear(64, latent_dim)  # Log-variance
        self.fc3 = nn.Linear(latent_dim, 64)
        self.fc4 = nn.Linear(64, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc21(h), self.fc22(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc3(z))
        return self.fc4(h)  # No activation, regression output

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


## Training VAE

In [8]:
def loss_function(recon_x, x, mu, logvar):
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_div

vae = VAE(input_dim, latent_dim)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(100):
    vae.train()
    train_loss = 0
    for batch in dataloader:
        x_batch = batch[0]
        optimizer.zero_grad()
        recon_batch, mu, logvar = vae(x_batch)
        loss = loss_function(recon_batch, x_batch, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {train_loss:.2f}")


Epoch 1, Loss: 6136.57
Epoch 2, Loss: 5922.29
Epoch 3, Loss: 5885.19
Epoch 4, Loss: 5881.05
Epoch 5, Loss: 5818.95
Epoch 6, Loss: 5704.84
Epoch 7, Loss: 5714.56
Epoch 8, Loss: 5640.28
Epoch 9, Loss: 5625.11
Epoch 10, Loss: 5577.67
Epoch 11, Loss: 5491.03
Epoch 12, Loss: 5545.53
Epoch 13, Loss: 5497.37
Epoch 14, Loss: 5486.97
Epoch 15, Loss: 5489.19
Epoch 16, Loss: 5456.21
Epoch 17, Loss: 5512.14
Epoch 18, Loss: 5456.40
Epoch 19, Loss: 5444.87
Epoch 20, Loss: 5393.89
Epoch 21, Loss: 5379.37
Epoch 22, Loss: 5414.37
Epoch 23, Loss: 5323.27
Epoch 24, Loss: 5386.58
Epoch 25, Loss: 5481.56
Epoch 26, Loss: 5324.50
Epoch 27, Loss: 5368.53
Epoch 28, Loss: 5338.08
Epoch 29, Loss: 5334.76
Epoch 30, Loss: 5402.74
Epoch 31, Loss: 5358.14
Epoch 32, Loss: 5303.18
Epoch 33, Loss: 5343.10
Epoch 34, Loss: 5261.59
Epoch 35, Loss: 5300.83
Epoch 36, Loss: 5392.67
Epoch 37, Loss: 5253.33
Epoch 38, Loss: 5220.17
Epoch 39, Loss: 5271.40
Epoch 40, Loss: 5291.94
Epoch 41, Loss: 5255.36
Epoch 42, Loss: 5283.33
E

## Anomaly Detection

In [6]:
vae.eval()
with torch.no_grad():
    x_recon, mu, logvar = vae(X_tensor)
    recon_error = torch.sum((x_recon - X_tensor) ** 2, dim=1)
    anomaly_scores = recon_error.numpy()

# Optional thresholding
threshold = np.percentile(anomaly_scores, 95)  # top 5% as anomalies
anomalies = anomaly_scores > threshold
print(f"Detected {np.sum(anomalies)} anomalies")


Detected 50 anomalies
